In [9]:
import os
import re
import pandas as pd
from fontTools.tfmLib import VANILLA
from networkx.algorithms.bipartite.basic import color

models = ['sasrec', 'caser', 'gru']
combs = [[1,1,1],[1,1,0],[1,0,1],[0,1,1],[0,1,0],[0,0,1]]
model_stop = ["main", "target"]
datasets = ["rc15_results", "retail_rocket_results"]
# MODIFY PATH TO div4rec dir of Paparellas
div4rec_path = "/home/marek/Kinit/MORSs/SMORL/div4rec"
results_dir = "/home/marek/Kinit/my_smorl/Plots/Paparella_both_pub/"
os.makedirs(results_dir, exist_ok=True)


#### Paparellas evaluation points and helper functions

In [12]:
points = {
    'rc15_results': {
        'gru': { 
            'base': 10000,
            '111': 10000,
            '001': 25000,
            '010': 25000,
            '011': 30000,
            '110': 25000,
            '101': 25000
        },
        'caser': {
            'base': 10000,
            '111': 75000,
            '001': 90000,
            '010': 40000,
            '011': 30000,
            '110': 110000,
            '101': 130000    
        },
        'sasrec': {
            'base': 25000,
            '111': 105000,
            '001': 115000,
            '010': 130000,
            '011': 115000,
            '110': 165000,
            '101': 115000  
        },
        'nextitnet': {
            'base': 10000,
            '111': 25000,
        }
    },
    'retail_rocket_results': {
        'gru': { 
            'base': 10000,
            '111': 30000,
            '001': 20000,
            '010': 20000,
            '011': 20000,
            '110': 30000,
            '101': 20000
        },
        'caser': {
            'base': 20000,
            '111': 50000,
            '001': 40000,
            '010': 50000,
            '011': 70000,
            '110': 40000,
            '101': 50000    
        },
        'sasrec': {
            'base': 10000,
            '111': 30000,
            '001': 30000,
            '010': 30000,
            '011': 30000,
            '110': 30000,
            '101': 30000  
        },
        'nextitnet': {
            'base': 10000,
            '111': 50000,
        }
    }
}

def get_max(dffile, testorval, alt=False):
    if alt:
        max_hr = dffile['ndcg_val_10'].max()
        threshold = 0.9 * max_hr
        filtered = dffile[dffile['ndcg_val_10'] >= threshold]
        max_row = filtered.loc[filtered['cov_val_10'].idxmax()]
    else:
        filtered = dffile[dffile['steps'] > 0]
        max_row = filtered.loc[filtered['ndcg_val_10'].idxmax()]
    
    max_step = max_row['steps']
    max_ndcg = max_row[f'ndcg_{testorval}_10']
    hr = max_row[f'hr_{testorval}_10']
    cov = max_row[f'cov_{testorval}_10']
    nov = max_row[f'nov_{testorval}_10']
    rep = max_row[f'rep_{testorval}_10']
    values = [max_step, max_ndcg, hr, cov, nov, rep]
    return values 

def compare_values(vanilla, rl):
    winners = []
    vanilla = vanilla[1:]
    rl = rl[1:]
    for i in range(len(vanilla)-1):
        if vanilla[i] > rl[i]:
            winners.append('BM')
        else:
            winners.append('RL')
    if vanilla[-1] < rl[-1]:
        winners.append('BM')
    else:
        winners.append('RL')
    return winners
    

#### Plots portrait format
#### Comparison of Vanilla and RL (various weights set)
#### ORIGINAL FORMAT
Following section produces plots in .png format, for example:

rc15_results-caser-main-test.png  
rc15_results-sasrec-target-val.png

In [77]:
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.cm as cm
import scienceplots
#print(plt.style.available)
#plt.style.use(['science', 'no-latex'])
#plt.subplots_adjust(
#    wspace=1.6,  # horizontal space between subplots
#    hspace=1.8,  # vertical space between subplots
#)

grey = cm.get_cmap('Greys')
green = cm.get_cmap('Greens')

def plot_helper(ax, df0, dfs, df6, PLOT, col1, col2, window=1):
    yname = ylab_helper(PLOT)
    ax.set_ylabel(yname, color=col1)
    ax.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color=col1, linestyle=":", linewidth=2)
    #for frame in dfs:
    #    ax.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color=col2, linewidth=1)
    if len(dfs) > 0:
        ax.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color=col2(0.2), linewidth=1)
        ax.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color=col2(0.3), linewidth=1)
        ax.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color=col2(0.4), linewidth=1)
        ax.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color=col2(0.5), linewidth=1)
        ax.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color=col2(0.6), linewidth=1)
    ax.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color=col1, linewidth=1)
    
def ylab_helper(name):
    if name.startswith("hr_") and name.endswith("_10"):
        label = 'HR@10'
    elif name.startswith("ndcg_") and name.endswith("_10"):
        label = 'nDCG@10'
    elif name.startswith("cov_") and name.endswith("_10"):
        label = 'COV@10'
    elif name.startswith("nov_") and name.endswith("_10"):
        label = 'NOV@10'
    elif name.startswith("rep_") and name.endswith("_5"):
        label = 'REP@5'
    elif name == "smorl":
        label = 'RL-head loss'
    elif name == "plain":
        label = 'self-supervised loss'
    else:
        label = name
    return label
    

def plot_metrics(basepath, dataset, model, replica, testorval, results_dir, window=1):
    filepath = f"{basepath}/{dataset}/{model}"
    dataline = "metrics"
    df0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    try:
        df1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
        df2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
        df3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
        df4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
        df5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
        dfs = [df1, df2, df3, df4, df5]
    except:
        dfs = []
    df6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}")
    
    alt_base = get_max(df0, testorval, alt=True)
    base = get_max(df0, testorval, alt=False)
    alt_rl =  get_max(df6, testorval, alt=True)
    rl = get_max(df6, testorval, alt=False)
    win1 = compare_values(base, rl)
    win2 = compare_values(alt_base, rl)
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(10, 10))
    fig.suptitle(f'Paparella : {dataset}-{model}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    #ax1.set_title("COV 10 / NOV 10", fontsize=10, pad=10)
    ax1.set_ylim(0.1, 0.75)
    yname = ylab_helper(PLOT)
    ax1.set_ylabel(yname, color="black")
    ax1.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label="vanilla")
    #for frame in dfs:
        #ax1.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgray', linewidth=1)
    if len(dfs) > 0:
        ax1.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color='0.2', linewidth=1)
        ax1.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color='0.3', linewidth=1)
        ax1.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color='0.4', linewidth=1)
        ax1.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color='0.5', linewidth=1)
        ax1.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color='0.6', linewidth=1)
    ax1.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color="black", linewidth=1, label="RL")
    
    PLOT = f"nov_{testorval}_10"
    ax2 = ax1.twinx()
    ax2.set_ylim(0.1, 0.75)
    plot_helper(ax2, df0, dfs, df6, PLOT, 'green', green)
       
    #~~~~~~ Plot cov + nov aligned ~~~~~~
    PLOT = f"cov_{testorval}_10"
    ax3 = axs[0, 1]
    #ax3.set_title("COV 10 / NOV 10 Alignment", fontsize=10, pad=10)
    yname = ylab_helper(PLOT)
    ax3.set_ylabel(yname, color='black')
    ax3.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2)
    #for frame in dfs:
    #    ax3.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgray', linewidth=4)
    if len(dfs) > 0:
        ax3.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color='0.2', linewidth=1)
        ax3.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color='0.3', linewidth=1)
        ax3.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color='0.4', linewidth=1)
        ax3.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color='0.5', linewidth=1)
        ax3.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color='0.6', linewidth=1)
    ax3.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='lightgray', linewidth=4)

    PLOT = f"nov_{testorval}_10"
    ax4 = ax3.twinx()
    plot_helper(ax4, df0, dfs, df6, PLOT, 'green', green)
    
    #~~~~~~ Plot hr + ndcg ~~~~~~   
    PLOT = f"hr_{testorval}_10"
    ax5 = axs[1, 0]
    #ax5.set_title("HR 10 / NDCG 10", fontsize=10, pad=10)
    plot_helper(ax5, df0, dfs, df6, PLOT, 'black', grey)
        
    PLOT = f"ndcg_{testorval}_10"
    ax6 = ax5.twinx()
    plot_helper(ax6, df0, dfs, df6, PLOT, 'green', green) 
        
    # ~~~~~~ Plot repetitivness ~~~~~~
    PLOT = f"rep_{testorval}_5"
    ax7 = axs[1, 1]
    #ax7.set_title("REP 5", fontsize=10, pad=10)
    ax7.yaxis.set_label_position("right")
    ax7.yaxis.tick_right()  
    plot_helper(ax7, df0, dfs, df6, PLOT, 'black', grey)
    
    # ~~~~~~ Plot losses ~~~~~~
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    try:
        dfl1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
        dfl2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
        dfl3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
        dfl4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
        dfl5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
        dfls = [dfl1, dfl2, dfl3, dfl4, dfl5]
    except:
        dfls = []
    dfl6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}") 
    
    PLOT = f"loss"
    ax8a = axs[2, 0]
    #ax8a.set_title("Rewards", fontsize=10, pad=10)
    ax8a.yaxis.set_label_position("left")
    ax8a.yaxis.tick_left()
    ax8a.set_ylim(0, 10)
    plot_helper(ax8a, dfl0, dfls, dfl6, PLOT, 'black', grey)
          
    PLOT = "plain"
    ax9 = axs[2, 1]
    #ax9.set_title("Loss components", fontsize=10, pad=10)
    ax9.set_ylim(0, 10)
    yname = ylab_helper(PLOT)
    ax9.set_ylabel(yname, color='green')
    for frame in dfls:
        ax9.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgray', linewidth=1)
    ax9.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='black', linewidth=1)

    PLOT = "smorl"
    ax10 = ax9.twinx()
    yname = ylab_helper(PLOT)
    ax10.set_ylabel(yname, color='green')
    ax10.set_ylim(0, 10)
    for frame in dfls:
        ax10.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgreen', linewidth=1)
    ax10.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='green', linewidth=1)
    
    
    for i, ax in enumerate(axs.flat):
        ax.axvline(x=points[dataset][model]['111'], color='blue', linewidth=0.5, label="evaluation point RL" if i == 0 else None)
        ax.axvline(x=points[dataset][model]['base'], color='red', linewidth=0.5, label="evaluation point vanilla" if i == 0 else None)
        ax.axvline(x=alt_rl[0], color='blue', linestyle=":", linewidth=0.5, label="alternative point RL" if i == 0 else None)
        ax.axvline(x=alt_base[0], color='red', linestyle=":", linewidth=0.5, label="alternative point vanilla" if i == 0 else None)
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
        ax.tick_params(axis='x', labelsize=8)
        for child in ax.figure.axes:
            child.tick_params(axis='y', labelsize=8)
            child.tick_params(axis='x', labelsize=8)
    
    fig.legend(
        loc='lower center',
        ncol=2,                  # number of columns in legend
        bbox_to_anchor=(0.5, 0.0),  # center below figure
        fontsize='medium'
    )
    fig.text(0.1, -0.06, 
             f"Vanilla_acc : NDCG: {base[1]:5.3f}, HR: {base[2]:5.3f}, COV: {base[3]:5.3f}, NOV: {base[4]:5.3f}, REP: {base[5]:5.3f}\nVanilla_opt : NDCG: {alt_base[1]:5.3f}, HR: {alt_base[2]:5.3f}, COV: {alt_base[3]:5.3f}, NOV: {alt_base[4]:5.3f}, REP: {alt_base[5]:5.3f}\nSMORL4RS_acc: NDCG: {rl[1]:5.3f}, HR: {rl[2]:5.3f}, COV: {rl[3]:5.3f}, NOV: {rl[4]:5.3f}, REP: {rl[5]:5.3f} \nSMORL4RS_opt: NDCG: {alt_rl[1]:5.3f}, HR: {alt_rl[2]:5.3f}, COV: {alt_rl[3]:5.3f}, NOV: {alt_rl[4]:5.3f}, REP: {alt_rl[5]:5.3f}", ha='left', fontsize=10)
    #fig.text(0.9, -0.06, f"Winners: ndcg, hr, cov, nov, rep: \n Winners base: {win1}\n Winners AltB: {win2}", ha='right', fontsize=10)
    
    plt.subplots_adjust(wspace=0.3, hspace=0.2)

    fig.savefig(f"{results_dir}{dataset}-{model}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

models = ["gru", "caser", "sasrec", "nextitnet"]
basepath = "/home/marek/Kinit/my_smorl/Plots/Paparella_both_new/"
for dataset in datasets:
    for model in models: 
        for replica in ['main', 'target']:
            for variant in ['test', 'val']:
                plot_metrics(basepath, dataset, model, replica, variant, results_dir)
            

/tmp/ipykernel_1721310/908104501.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  grey = cm.get_cmap('Greys')
/tmp/ipykernel_1721310/908104501.py:13: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  green = cm.get_cmap('Greens')


#### SIMPLYFIED FORMAT

In [117]:
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.cm as cm
import scienceplots
#print(plt.style.available)
#plt.style.use(['science', 'no-latex'])
#plt.subplots_adjust(
#    wspace=1.6,  # horizontal space between subplots
#    hspace=1.8,  # vertical space between subplots
#)

grey = cm.get_cmap('Greys')
green = cm.get_cmap('Greens')

def plot_helper(ax, df0, dfs, df6, PLOT, col1, col2, window=1):
    yname = ylab_helper(PLOT)
    ax.set_ylabel(yname, color=col1)
    PLOT1 = PLOT
    if PLOT != 'smorl':
        if PLOT == 'plain':
            PLOT1 = 'loss'
        ax.plot(df0['steps'], df0[PLOT1].rolling(window).mean(), color=col1, linestyle=":", linewidth=2)
    #for frame in dfs:
    #    ax.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color=col2, linewidth=1)
    if len(dfs) > 0:
        ax.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color=col2(0.2), linewidth=1)
        ax.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color=col2(0.3), linewidth=1)
        ax.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color=col2(0.4), linewidth=1)
        ax.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color=col2(0.5), linewidth=1)
        ax.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color=col2(0.6), linewidth=1)
    ax.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color=col1, linewidth=1) 

def plot_metrics(basepath, dataset, model, replica, testorval, results_dir, window=1):
    filepath = f"{basepath}/{dataset}/{model}"
    dataline = "metrics"
    df0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    try:
        df1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
        df2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
        df3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
        df4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
        df5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
        dfs = [df1, df2, df3, df4, df5]
    except:
        dfs = []
    df6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}")
    
    alt_base = get_max(df0, testorval, alt=True)
    base = get_max(df0, testorval, alt=False)
    alt_rl =  get_max(df6, testorval, alt=True)
    rl = get_max(df6, testorval, alt=False)
    win1 = compare_values(base, rl)
    win2 = compare_values(alt_base, rl)
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(10, 10), sharex=True)
    fig.suptitle(f'Paparella : {dataset}-{model}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    #ax1.set_title("COV 10 / NOV 10", fontsize=10, pad=10)
    ax1.set_ylim(0.1, 0.75)
    yname = ylab_helper(PLOT)
    ax1.set_ylabel(yname, color="black")
    ax1.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label="Vanilla")
    #for frame in dfs:
        #ax1.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgray', linewidth=1)
    if len(dfs) > 0:
        ax1.plot(dfs[0]['steps'], dfs[0][PLOT].rolling(window).mean(), color=grey(0.2), linewidth=1, label="SMORL4RS[0.0.1]")
        ax1.plot(dfs[1]['steps'], dfs[1][PLOT].rolling(window).mean(), color=grey(0.3), linewidth=1, label="SMORL4RS[0.1.0]")
        ax1.plot(dfs[2]['steps'], dfs[2][PLOT].rolling(window).mean(), color=grey(0.4), linewidth=1, label="SMORL4RS[0.1.1]")
        ax1.plot(dfs[3]['steps'], dfs[3][PLOT].rolling(window).mean(), color=grey(0.5), linewidth=1, label="SMORL4RS[1.1.0]")
        ax1.plot(dfs[4]['steps'], dfs[4][PLOT].rolling(window).mean(), color=grey(0.6), linewidth=1, label="SMORL4RS[1.0.1]")
    ax1.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color="black", linewidth=1, label="SMORL4RS[1.1.1]")
    
    PLOT = f"nov_{testorval}_10"
    ax2 = axs[0, 1]
    ax2.set_ylim(0.1, 0.75)
    yname = ylab_helper(PLOT)
    ax2.set_ylabel(yname, color='black')
    ax2.yaxis.set_label_position("right")
    ax2.yaxis.tick_right()  

    plot_helper(ax2, df0, dfs, df6, PLOT, 'black', grey)
          
    #~~~~~~ Plot hr + ndcg ~~~~~~   
    PLOT = f"hr_{testorval}_10"
    ax5 = axs[1, 0]
    #ax5.set_title("HR 10 / NDCG 10", fontsize=10, pad=10)
    plot_helper(ax5, df0, dfs, df6, PLOT, 'black', grey)
        
    PLOT = f"ndcg_{testorval}_10"
    ax6 = axs[1,1]
    ax6.yaxis.set_label_position("right")
    ax6.yaxis.tick_right()  
    plot_helper(ax6, df0, dfs, df6, PLOT, 'black', grey) 
    
    # ~~~~~~ Plot losses ~~~~~~
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    try:
        dfl1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
        dfl2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
        dfl3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
        dfl4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
        dfl5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
        dfls = [dfl1, dfl2, dfl3, dfl4, dfl5]
    except:
        dfls = []
    dfl6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}") 
          
    PLOT = "plain"
    ax9 = axs[2, 0]
    #ax9.set_title("Loss components", fontsize=10, pad=10)
    ax9.set_ylim(0, 10)
    yname = ylab_helper(PLOT)
    ax9.set_ylabel(yname, color='black')  
    ax9.set_xlabel("Steps", color='black')
    plot_helper(ax9, dfl0, dfls, dfl6, PLOT, 'black', grey, window=5)

    PLOT = "smorl"
    ax10 = axs[2, 1]
    yname = ylab_helper(PLOT)
    ax10.set_ylabel(yname, color='black')
    ax10.set_xlabel("Steps", color='black')
    ax10.yaxis.set_label_position("right")
    ax10.yaxis.tick_right() 
    ax10.set_ylim(0, 10)
    plot_helper(ax10, dfl0, dfls, dfl6, PLOT, 'black', grey, window=5)
    
    
    for i, ax in enumerate(axs.flat):
        ax.axvline(x=points[dataset][model]['111'], color='blue', linewidth=0.5, label="evaluation point RL" if i == 0 else None)
        ax.axvline(x=points[dataset][model]['base'], color='red', linewidth=0.5, label="evaluation point vanilla" if i == 0 else None)
        ax.axvline(x=alt_rl[0], color='blue', linestyle=":", linewidth=0.5, label="alternative point RL" if i == 0 else None)
        ax.axvline(x=alt_base[0], color='red', linestyle=":", linewidth=0.5, label="alternative point vanilla" if i == 0 else None)
        #ax.axvline(x=max_step_RL, color='red', linewidth=0.5, label="alternative_max_RL" if i == 0 else None)  # Only label on first plot
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
        ax.tick_params(axis='x', labelsize=8)
        for child in ax.figure.axes:
            child.tick_params(axis='y', labelsize=8)
            child.tick_params(axis='x', labelsize=8)
    
    fig.legend(
        loc='lower center',
        ncol=4,                  # number of columns in legend
        bbox_to_anchor=(0.5, -0.02),  # center below figure
        fontsize='medium'
    )
    fig.text(0.2, -0.1, 
             f"Vanilla acc : NDCG: {base[1]:5.3f}, HR: {base[2]:5.3f}, COV: {base[3]:5.3f}, NOV: {base[4]:5.3f}, REP: {base[5]:5.3f}\nVanilla opt : NDCG: {alt_base[1]:5.3f}, HR: {alt_base[2]:5.3f}, COV: {alt_base[3]:5.3f}, NOV: {alt_base[4]:5.3f}, REP: {alt_base[5]:5.3f}\nSMORL4RS acc: NDCG: {rl[1]:5.3f}, HR: {rl[2]:5.3f}, COV: {rl[3]:5.3f}, NOV: {rl[4]:5.3f}, REP: {rl[5]:5.3f} \nSMORL4RS opt: NDCG: {alt_rl[1]:5.3f}, HR: {alt_rl[2]:5.3f}, COV: {alt_rl[3]:5.3f}, NOV: {alt_rl[4]:5.3f}, REP: {alt_rl[5]:5.3f}", ha='left', fontsize=10)
    #fig.text(0.9, -0.06, f"Winners: ndcg, hr, cov, nov, rep: \n Winners base: {win1}\n Winners AltB: {win2}", ha='right', fontsize=10)
    
    plt.subplots_adjust(wspace=0.1, hspace=0.1)

    fig.savefig(f"{results_dir}Simple-{dataset}-{model}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

models = ["gru", "caser", "sasrec", "nextitnet"]
basepath = "/home/marek/Kinit/my_smorl/Plots/Paparella_both_new/"
for dataset in datasets:
    for model in models: 
        for replica in ['main', 'target']:
            for variant in ['test', 'val']:
                plot_metrics(basepath, dataset, model, replica, variant, results_dir)
            

/tmp/ipykernel_1721310/1112023526.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  grey = cm.get_cmap('Greys')
/tmp/ipykernel_1721310/1112023526.py:13: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  green = cm.get_cmap('Greens')


#### Comparison of individual models
Following section produces plots to compare models as:  

COMPARE_MODELS-rc15_results-main-test.png

In [132]:
def plot_helper(ax, df_base_s, df_rl_s, df_base_c, df_rl_c,df_base_g, df_rl_g, df_base_n, df_rl_n, PLOT, window=1):
    yname = ylab_helper(PLOT)
    ax.set_ylabel(yname, color='black')
    ax.plot(df_base_s['steps'], df_base_s[PLOT].rolling(window).mean(), color='blue', linestyle=":", linewidth=2, label='SASRec')
    ax.plot(df_rl_s['steps'], df_rl_s[PLOT].rolling(window).mean(), color='blue', linewidth=1, label='SMORL4RS-SASRec')
    ax.plot(df_base_c['steps'], df_base_c[PLOT].rolling(window).mean(), color='green', linestyle=":", linewidth=2, label='Caser')
    ax.plot(df_rl_c['steps'], df_rl_c[PLOT].rolling(window).mean(), color='green', linewidth=1, label='SMORL4RS-Caser')
    ax.plot(df_base_g['steps'], df_base_g[PLOT].rolling(window).mean(), color='red', linestyle=":", linewidth=2, label='GRU')
    ax.plot(df_rl_g['steps'], df_rl_g[PLOT].rolling(window).mean(), color='red', linewidth=1, label='SMORL4RS-GRU')
    ax.plot(df_base_n['steps'], df_base_n[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label='NextItNet')
    ax.plot(df_rl_n['steps'], df_rl_n[PLOT].rolling(window).mean(), color='black', linewidth=1, label='SMORL4RS-NextItNet')

def plot_metrics(basepath, dataset, replica, testorval, results_dir, window=1):  
    filepath = f"{basepath}/{dataset}"
    dataline = "metrics"
    
    df_base_s = pd.read_pickle(f"{filepath}/sasrec/base_{dataline}")
    df_rl_s = pd.read_pickle(f"{filepath}/sasrec/rl_111_{replica}_{dataline}")
    df_base_c = pd.read_pickle(f"{filepath}/caser/base_{dataline}")
    df_rl_c = pd.read_pickle(f"{filepath}/caser/rl_111_{replica}_{dataline}")
    df_base_g = pd.read_pickle(f"{filepath}/gru/base_{dataline}")
    df_rl_g = pd.read_pickle(f"{filepath}/gru/rl_111_{replica}_{dataline}")
    df_base_n = pd.read_pickle(f"{filepath}/nextitnet/base_{dataline}")
    df_rl_n = pd.read_pickle(f"{filepath}/nextitnet/rl_111_{replica}_{dataline}")
        
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle(f'Paparella : {dataset}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    #ax1.set_title("COV 10", fontsize=10, pad=10)
    ax1.set_ylim(0, 0.8)
    ax1.set_ylabel(PLOT, color='black')
    plot_helper(ax1, df_base_s, df_rl_s, df_base_c, df_rl_c,df_base_g, df_rl_g, df_base_n, df_rl_n, PLOT)
    
    PLOT = f"nov_{testorval}_10"
    ax2 = axs[0, 1]
    ax2.set_ylim(0, 0.8)
    #ax2.set_title("NOV 10", fontsize=10, pad=10)
    plot_helper(ax2, df_base_s, df_rl_s, df_base_c, df_rl_c,df_base_g, df_rl_g, df_base_n, df_rl_n, PLOT)
    ax2.yaxis.set_label_position("right")
    ax2.yaxis.tick_right()  
   
    PLOT = f"hr_{testorval}_10"
    ax3 = axs[1, 0]
    ax3.set_ylim(0.2, 0.45)
    #ax3.set_title("HR 10", fontsize=10, pad=10)
    plot_helper(ax3, df_base_s, df_rl_s, df_base_c, df_rl_c,df_base_g, df_rl_g, df_base_n, df_rl_n, PLOT)
    
    PLOT = f"ndcg_{testorval}_10"
    ax4 = axs[1, 1]
    #ax4.set_title("NDCG 10", fontsize=10, pad=10)
    ax4.set_ylim(0.15, 0.30)
    plot_helper(ax4, df_base_s, df_rl_s, df_base_c, df_rl_c,df_base_g, df_rl_g, df_base_n, df_rl_n, PLOT) 
    ax4.yaxis.set_label_position("right")
    ax4.yaxis.tick_right()  
    
    PLOT = f"rep_{testorval}_5"
    ax5 = axs[2, 0]
    ax5.set_ylim(5, 20)
    #ax5.set_title("REP 5", fontsize=10, pad=10)
    plot_helper(ax5, df_base_s, df_rl_s, df_base_c, df_rl_c,df_base_g, df_rl_g, df_base_n, df_rl_n, PLOT) 
    ax5.set_xlabel("Steps", color='black')
    
    # plot Loss
    dataline = "loss"
    dfl_base_s = pd.read_pickle(f"{filepath}/sasrec/base_{dataline}")
    dfl_rl_s = pd.read_pickle(f"{filepath}/sasrec/rl_111_{replica}_{dataline}")
    dfl_base_c = pd.read_pickle(f"{filepath}/caser/base_{dataline}")
    dfl_rl_c = pd.read_pickle(f"{filepath}/caser/rl_111_{replica}_{dataline}")
    dfl_base_g = pd.read_pickle(f"{filepath}/gru/base_{dataline}")
    dfl_rl_g = pd.read_pickle(f"{filepath}/gru/rl_111_{replica}_{dataline}")
    dfl_base_n = pd.read_pickle(f"{filepath}/nextitnet/base_{dataline}")
    dfl_rl_n = pd.read_pickle(f"{filepath}/nextitnet/rl_111_{replica}_{dataline}")
    
    PLOT = f"loss"
    ax6 = axs[2, 1]
    ax6.set_ylim(3, 10)
    #ax6.set_title("Loss", fontsize=10, pad=10)
    plot_helper(ax6, dfl_base_s, dfl_rl_s, dfl_base_c, dfl_rl_c, dfl_base_g, dfl_rl_g, dfl_base_n, dfl_rl_n, PLOT, window=10) 
    ax6.set_xlabel("Steps", color='black')
    ax6.yaxis.set_label_position("right")
    ax6.yaxis.tick_right()  
    
    for i, ax in enumerate(axs.flat):
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=10)
        ax.tick_params(axis='x', labelsize=10)
    
    handles, labels = ax1.get_legend_handles_labels()
    seen = set()
    unique = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]
    fig.legend(*zip(*unique), loc='lower center', ncol=4, bbox_to_anchor=(0.5, 0.03), fontsize='medium')
    fig.savefig(f"{results_dir}COMPARE-MODELS-{dataset}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

basepath = "/home/marek/Kinit/my_smorl/Plots/Paparella_both/"
for dataset in datasets:
    for replica in ['main', 'target']:
        for variant in ['test', 'val']:
            plot_metrics(basepath, dataset, replica, variant, results_dir)
            

#### Plots simplified for publication

In [128]:
import pandas as pd
from matplotlib import pyplot as plt

def plot_helper(ax, df0, df6, PLOT, col1, col2, window=1):
    yname = ylab_helper(PLOT)
    ax.set_ylabel(yname, color=col1, size=12)
    ax.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color=col1, linestyle=":", linewidth=2)
    ax.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color=col1, linewidth=1)
    

def plot_metrics(basepath, dataset, model, replica, testorval, results_dir, window=1):
    filepath = f"{basepath}/{dataset}/{model}"
    dataline = "metrics"
    df0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    df6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}")
    
    alt_base = get_max(df0, testorval, alt=True)
    base = get_max(df0, testorval, alt=False)
    alt_rl = get_max(df6, testorval, alt=True)
    rl = get_max(df6, testorval, alt=False)
    #win1 = compare_values(base, rl)
    #win2 = compare_values(alt_base, rl)
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 1, figsize=(8, 8))
    #fig.suptitle(f'Paparella : {dataset}-{model}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[1]
    #ax1.set_title("COV 10 / NOV 10", fontsize=15, pad=10)
    ax1.set_ylim(0.1, 0.75)
    yname = ylab_helper(PLOT)
    ax1.set_ylabel(yname, color="black", size=12)
    ax1.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label="vanilla")
    ax1.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color="black", linewidth=1, label="RL")
    
    PLOT = f"nov_{testorval}_10"
    ax2 = ax1.twinx()
    ax2.set_ylim(0.1, 0.75)
    plot_helper(ax2, df0, df6, PLOT, 'green', 'lightgreen')
        
    #~~~~~~ Plot hr + ndcg ~~~~~~   
    PLOT = f"hr_{testorval}_10"
    ax5 = axs[0]
    #ax5.set_title("HR 10 / NDCG 10", fontsize=15, pad=10)
    plot_helper(ax5, df0, df6, PLOT, 'black', 'lightgray')
        
    PLOT = f"ndcg_{testorval}_10"
    ax6 = ax5.twinx()
    plot_helper(ax6, df0, df6, PLOT, 'green', 'lightgreen') 
        
    # ~~~~~~ Plot repetitivness ~~~~~~
    '''
    PLOT = f"rep_{testorval}_5"
    ax7 = axs[1, 0]
    ax7.set_title("REP 5", fontsize=15, pad=10)
    ax7.yaxis.set_label_position("left")
    ax7.yaxis.tick_right()  
    plot_helper(ax7, df0, df6, PLOT, 'black', 'lightgray')
    '''
    
    # ~~~~~~ Plot losses ~~~~~~
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    dfl6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}") 
          
    PLOT = "plain"
    ax9 = axs[2]
    #ax9.set_title("Loss components", fontsize=15, pad=10)
    ax9.set_ylim(0, 7)
    yname = ylab_helper(PLOT)
    ax9.set_ylabel(yname, color="black", size=12)
    ax9.set_xlabel("Steps", color="black", size=12)
    ax9.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='black', linewidth=1)
    ax9.plot(dfl0['steps'], dfl0['loss'].rolling(5).mean(), color='black', linestyle=":", linewidth=2)

    PLOT = "smorl"
    ax10 = ax9.twinx()
    yname = ylab_helper(PLOT)
    ax10.set_ylabel(yname, color='green', size=12)
    ax10.set_ylim(0, 7)
    ax10.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='green', linewidth=1)
    
    
    for i, ax in enumerate(axs.flat):
        ax.axvline(x=points[dataset][model]['111'], color='blue', linewidth=0.5, label="evaluation point RL" if i == 0 else None)
        ax.axvline(x=points[dataset][model]['base'], color='red', linewidth=0.5, label="evaluation point vanilla" if i == 0 else None)
        ax.axvline(x=alt_rl[0], color='blue', linestyle=":", linewidth=0.5, label="alternative point RL" if i == 0 else None)
        ax.axvline(x=alt_base[0], color='red', linestyle=":", linewidth=0.5, label="alternative point vanilla" if i == 0 else None)
        #ax.axvline(x=max_step_RL, color='red', linewidth=0.5, label="alternative_max_RL" if i == 0 else None)  # Only label on first plot
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=10)
        ax.tick_params(axis='x', labelsize=10)
        for child in ax.figure.axes:
            child.tick_params(axis='y', labelsize=10)
            child.tick_params(axis='x', labelsize=10)
        
    
    fig.legend(
        loc='lower center',
        ncol=3,                  # number of columns in legend
        bbox_to_anchor=(0.5, -0.08),  # center below figure
        fontsize='large'
    )
    
    
    
    fig.text(0.2, -0.2, 
             f"Vanilla acc : NDCG: {base[1]:5.3f}, HR: {base[2]:5.3f}, COV: {base[3]:5.3f}, NOV: {base[4]:5.3f}, REP: {base[5]:5.3f}\nVanilla opt : NDCG: {alt_base[1]:5.3f}, HR: {alt_base[2]:5.3f}, COV: {alt_base[3]:5.3f}, NOV: {alt_base[4]:5.3f}, REP: {alt_base[5]:5.3f}\nSMORL4RS acc: NDCG: {rl[1]:5.3f}, HR: {rl[2]:5.3f}, COV: {rl[3]:5.3f}, NOV: {rl[4]:5.3f}, REP: {rl[5]:5.3f}\nSMORL4RS opt: NDCG: {alt_rl[1]:5.3f}, HR: {alt_rl[2]:5.3f}, COV: {alt_rl[3]:5.3f}, NOV: {alt_rl[4]:5.3f}, REP: {alt_rl[5]:5.3f}", ha='left', fontsize=10)
    #fig.text(0.9, -0.06, f"Winners: ndcg, hr, cov, nov, rep: \n Winners base: {win1}\n Winners AltB: {win2}",
    #         ha='right', fontsize=10)
    
    plt.tight_layout()
    fig.savefig(f"{results_dir}/Pub/{dataset}-{model}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

models = ["gru", "caser", "sasrec", "nextitnet"]
basepath = "/home/marek/Kinit/my_smorl/Plots/Paparella_both/"
newpath = "/home/marek/Kinit/my_smorl/Plots/Paparella_both_pub/Pub"
os.makedirs(newpath, exist_ok=True)
for dataset in datasets:
    for model in models: 
        for replica in ['main', 'target']:
            for variant in ['test', 'val']:
                plot_metrics(basepath, dataset, model, replica, variant, results_dir)